# Model Merging for Multitask Language Models
#### Islem Khemissi [LinkedIn](https://ca.linkedin.com/in/islem-khemissi) 

### A Demonstration Notebook

**Context:** The full experiment was run on a workstation with 2× NVIDIA Quadro RTX 8000 (96 GB VRAM total), using `meta-llama/Llama-3.2-3B-Instruct` as the base model with full-scale datasets and multi-day compute. The full code is on [GitHub](https://github.com/islem-kms/COMP6861_MergeLLMs) and all fine-tuned adapters and merged models are available on [HuggingFace (islemkms)](https://huggingface.co/islemkms) (uploaded to free up server space).

For this demo we use `Qwen/Qwen2.5-0.5B-Instruct` (494 M parameters, publicly accessible without gating), 200 training samples per task, and 50 training steps per specialist. The **LoRA configuration and all merging algorithms are identical** to the full experiment.

**The notebook follows the actual experimental progression:**
1. **Part 1** — Fine-tune emotion and summarization specialists; run initial 2-task merges with all four methods
2. **Part 2** — Tune λ (Task Arithmetic) and density (Breadcrumbs) to find better 2-task configurations
3. **Part 3** — Add an NLI specialist (three tasks) to test whether TIES improves with a proper majority vote
4. **Part 4** — TIES parameter sweep to find the optimal scaling factor for 3-task merging

## Setup

Install dependencies and configure the experiment. The LoRA hyperparameters below (`r=16`, `lora_alpha=32`, same target modules) are **identical to the full experiment**, ensuring the methodology is faithfully reproduced even at smaller scale.

In [ ]:
%%capture
!pip install transformers peft trl accelerate datasets evaluate rouge_score

In [ ]:
import os, gc, warnings
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import evaluate as hf_evaluate

warnings.filterwarnings("ignore")

# ── Model ──────────────────────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE    = torch.bfloat16

# ── LoRA config (identical to full experiment) ─────────────────────────────────
LORA_R       = 16
LORA_ALPHA   = 32
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj"]
LORA_DROPOUT = 0.05

# ── Demo scale ─────────────────────────────────────────────────────────────────
MAX_STEPS = 50     # full experiment: 3 epochs (~4 000–8 000 steps)
TRAIN_N   = 200    # full experiment: up to 20 000 samples per task
EVAL_N    = 50     # full experiment: 500–1 000 samples
MAX_SEQ   = 128

# ── Label maps ─────────────────────────────────────────────────────────────────
EMOTION_LABELS = {0: "sadness", 1: "joy", 2: "love", 3: "anger", 4: "fear", 5: "surprise"}
NLI_LABELS     = {0: "entailment", 1: "neutral", 2: "contradiction"}

# ── Directories ────────────────────────────────────────────────────────────────
for d in ["adapters/emotion", "adapters/summary", "adapters/nli", "state_dicts", "results"]:
    os.makedirs(d, exist_ok=True)

print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Dataset Preparation

We use three datasets matching the full experiment:
- **Emotion classification** (`dair-ai/emotion`): 6-class emotion labeling (sadness, joy, love, anger, fear, surprise)
- **Dialogue summarization** (`knkarthick/dialogsum`): generate a short summary from a dialogue
- **Natural Language Inference** (`nyu-mll/multi_nli`): predict entailment / neutral / contradiction

Each sample is formatted as a plain-text prompt. The same prompt template is used at both training and evaluation time.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

# ── Emotion ────────────────────────────────────────────────────────────────────
raw_emotion      = load_dataset("dair-ai/emotion", "split")
emotion_train_ds = raw_emotion["train"].shuffle(seed=42).select(range(TRAIN_N))
emotion_eval_raw = raw_emotion["validation"].select(range(EVAL_N))

def fmt_emotion(ex):
    return {"text": f"Classify the emotion.\nText: {ex['text']}\nEmotion: {EMOTION_LABELS[ex['label']]}"}
emotion_train_ds = emotion_train_ds.map(fmt_emotion)

# ── Summarization ──────────────────────────────────────────────────────────────
raw_summary      = load_dataset("knkarthick/dialogsum")
summary_train_ds = raw_summary["train"].shuffle(seed=42).select(range(TRAIN_N))
summary_eval_raw = raw_summary["validation"].select(range(EVAL_N))

def fmt_summary(ex):
    return {"text": f"Summarize the dialogue.\nDialogue: {ex['dialogue']}\nSummary: {ex['summary']}"}
summary_train_ds = summary_train_ds.map(fmt_summary)

# ── NLI ────────────────────────────────────────────────────────────────────────
raw_nli      = load_dataset("nyu-mll/multi_nli")
nli_train_ds = raw_nli["train"].filter(lambda x: x["label"] != -1).shuffle(seed=42).select(range(TRAIN_N))
nli_eval_raw = raw_nli["validation_matched"].filter(lambda x: x["label"] != -1).select(range(EVAL_N))

def fmt_nli(ex):
    return {"text": f"Premise: {ex['premise']}\nHypothesis: {ex['hypothesis']}\nRelation: {NLI_LABELS[ex['label']]}"}
nli_train_ds = nli_train_ds.map(fmt_nli)

print(f"Emotion  train={len(emotion_train_ds)}  eval={len(emotion_eval_raw)}")
print(f"Summary  train={len(summary_train_ds)}  eval={len(summary_eval_raw)}")
print(f"NLI      train={len(nli_train_ds)}      eval={len(nli_eval_raw)}")

---
## Part 1: Two-Task Experiment

### Step 1 — Fine-tune Specialist Models

We fine-tune a separate LoRA adapter for each task using `SFTTrainer`. Each adapter starts from the same frozen base model and is trained **independently** — specialists never see each other's data. The goal is to later *merge* multitask capability from isolated specialists rather than training on mixed data.

LoRA (`r=16`, `lora_alpha=32`) decomposes each weight update as `ΔW = BA`, reducing trainable parameters from ~494 M to ~4.7 M per adapter (≈1 % of the model).

In [ ]:
def get_lora_config():
    return LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGETS,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

def finetune(train_ds, adapter_dir, max_steps=MAX_STEPS):
    if os.path.exists(os.path.join(adapter_dir, "adapter_model.safetensors")) or \
       os.path.exists(os.path.join(adapter_dir, "adapter_model.bin")):
        print(f"  Adapter already at {adapter_dir} — skipping.")
        return

    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map="auto")
    model.enable_input_require_grads()
    model = get_peft_model(model, get_lora_config())
    model.print_trainable_parameters()

    args = SFTConfig(
        output_dir=adapter_dir,
        max_steps=max_steps,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=1,
        learning_rate=2e-4,
        warmup_steps=5,
        bf16=torch.cuda.is_available(),
        fp16=False,
        logging_steps=10,
        save_strategy="no",
        dataset_text_field="text",
        max_seq_length=MAX_SEQ,
        dataloader_num_workers=0,
        report_to="none",
        packing=False,
    )
    trainer = SFTTrainer(model=model, train_dataset=train_ds, args=args)
    trainer.train()
    model.save_pretrained(adapter_dir)
    print(f"  Saved adapter → {adapter_dir}")

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
print("Fine-tuning EMOTION specialist...")
finetune(emotion_train_ds, "adapters/emotion")

In [ ]:
print("Fine-tuning SUMMARY specialist...")
finetune(summary_train_ds, "adapters/summary")

### Step 2 — Linearize Adapters

Before merging, we *fuse* each LoRA adapter back into the base model weights using `merge_and_unload()`, producing a full-rank state dict `θ_finetuned` for each specialist. The base model state dict `θ_base` is saved separately. All state dicts are kept in **float32** for merging precision.

In [ ]:
def linearize(adapter_dir, out_path):
    if os.path.exists(out_path):
        print(f"  {out_path} already exists — skipping.")
        return
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32, device_map="cpu")
    model = PeftModel.from_pretrained(model, adapter_dir)
    model = model.merge_and_unload()
    torch.save(model.state_dict(), out_path)
    del model; gc.collect()
    print(f"  Saved {out_path}")

if not os.path.exists("state_dicts/sd_base.pt"):
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32, device_map="cpu")
    torch.save(base.state_dict(), "state_dicts/sd_base.pt")
    del base; gc.collect()
    print("  Saved state_dicts/sd_base.pt")

linearize("adapters/emotion", "state_dicts/sd_emotion.pt")
linearize("adapters/summary", "state_dicts/sd_summary.pt")
print("Linearization complete.")

### Step 3 — Merging Methods

All methods operate in **parameter space** on the linearized state dicts. No gradient information is needed at merge time.

**Weight Averaging** — `θ_merged = (1/N) Σ θᵢ`. The simplest baseline: average all parameters directly. Works when models share a loss basin, but ignores task-specific structure.

**Task Arithmetic** — Extract a *task vector* `τᵢ = θᵢ − θ_base` for each specialist, then apply `θ_merged = θ_base + λ · Σ τᵢ`. The **scaling factor λ** controls how strongly each task vector is applied to the base model. λ=0 leaves the base unchanged; λ=1.0 applies the full sum of task vectors. Too small → under-merged (close to base); too large → task interference and performance collapse.

**Breadcrumbs (Sparse Task Arithmetic)** — Before summing, prune each task vector to keep only its top-**density** fraction of entries by absolute magnitude. The **density** parameter (0–1) controls sparsity: density=1.0 is identical to Task Arithmetic (no pruning); density=0.2 keeps only the top 20% of parameters. High-magnitude entries carry the task-specific signal; zeroing the rest reduces cross-task interference when task vectors are summed.

**TIES (Trim, Elect Sign)** — After trimming (same as Breadcrumbs), runs a **majority vote** to elect a sign for each parameter position across all task vectors. Only parameters that agree with the elected sign are averaged and applied. This directly resolves *sign conflicts* — cases where different tasks push the same weight in opposite directions — which is a fundamental failure mode of Task Arithmetic.

In [ ]:
def load_sd(path):
    return torch.load(path, map_location="cpu", weights_only=True)

def compute_task_vector(base_sd, ft_sd):
    return {k: ft_sd[k].float() - base_sd[k].float() for k in base_sd}

def sparsify(tv, density):
    sparse = {}
    for k, v in tv.items():
        if v.numel() > 1:
            thresh = v.abs().flatten().topk(max(1, int(v.numel() * density))).values.min()
            sparse[k] = v * (v.abs() >= thresh).float()
        else:
            sparse[k] = v
    return sparse

def weight_average(specialist_sds):
    keys = specialist_sds[0].keys()
    n    = len(specialist_sds)
    return {k: sum(sd[k].float() for sd in specialist_sds) / n for k in keys}

def task_arithmetic(base_sd, specialist_sds, lam=0.5):
    tvs = [compute_task_vector(base_sd, sd) for sd in specialist_sds]
    return {k: base_sd[k].float() + lam * sum(tv[k] for tv in tvs) for k in base_sd}

def breadcrumbs(base_sd, specialist_sds, lam=0.5, density=0.5):
    tvs = [sparsify(compute_task_vector(base_sd, sd), density) for sd in specialist_sds]
    return {k: base_sd[k].float() + lam * sum(tv[k] for tv in tvs) for k in base_sd}

def ties(base_sd, specialist_sds, lam=0.5, density=0.4):
    trimmed = [sparsify(compute_task_vector(base_sd, sd), density) for sd in specialist_sds]
    merged  = {}
    for k in base_sd:
        stacked   = torch.stack([tv[k].float() for tv in trimmed])
        sign_vote = stacked.sum(0).sign()
        sign_vote[sign_vote == 0] = 1
        agreed    = stacked * (stacked.sign() == sign_vote.unsqueeze(0)).float()
        count     = (agreed != 0).float().sum(0).clamp(min=1)
        merged_tv = agreed.sum(0) / count
        merged[k] = base_sd[k].float() + lam * merged_tv
    return merged

print("Merging functions defined.")

### Step 4 — Run Initial 2-Task Merges

We merge with default parameters as a starting point: λ=0.5 for all methods, density=0.5 for Breadcrumbs, density=0.4 for TIES.

In [ ]:
def run_2task_merges():
    base_sd    = load_sd("state_dicts/sd_base.pt")
    emotion_sd = load_sd("state_dicts/sd_emotion.pt")
    summary_sd = load_sd("state_dicts/sd_summary.pt")
    specialists = [emotion_sd, summary_sd]

    configs = [
        ("sd_weight_avg_2task.pt",  lambda: weight_average(specialists)),
        ("sd_task_arith_2task.pt",  lambda: task_arithmetic(base_sd, specialists, lam=0.5)),
        ("sd_breadcrumbs_2task.pt", lambda: breadcrumbs(base_sd, specialists, lam=0.5, density=0.5)),
        ("sd_ties_2task.pt",        lambda: ties(base_sd, specialists, lam=0.5, density=0.4)),
    ]
    for fname, merge_fn in configs:
        path = f"state_dicts/{fname}"
        if not os.path.exists(path):
            print(f"Merging → {fname}")
            sd = merge_fn()
            torch.save(sd, path)
            del sd; gc.collect()
        else:
            print(f"  {fname} already exists — skipping.")

    del base_sd, emotion_sd, summary_sd; gc.collect()
    print("2-task merges complete.")

run_2task_merges()

### Step 5 — Evaluate 2-Task Models

Each model is loaded from its state dict, cast to bfloat16, and evaluated with greedy decoding.
- **Emotion accuracy**: fraction of samples where the predicted label matches ground truth
- **ROUGE-L**: longest common subsequence recall between generated and reference summaries

In [ ]:
rouge_metric = hf_evaluate.load("rouge")

def load_model_from_sd(sd_path):
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map="auto")
    sd    = load_sd(sd_path)
    model.load_state_dict({k: v.to(DTYPE) for k, v in sd.items()})
    model.eval()
    del sd; gc.collect()
    return model

def generate(model, prompt, max_new_tokens=20):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def eval_emotion(model):
    correct = 0
    for ex in emotion_eval_raw:
        pred  = generate(model, f"Classify the emotion.\nText: {ex['text']}\nEmotion:", max_new_tokens=5).lower()
        correct += int(EMOTION_LABELS[ex["label"]] in pred)
    return round(correct / len(emotion_eval_raw), 3)

def eval_summary(model):
    preds, refs = [], []
    for ex in summary_eval_raw:
        preds.append(generate(model, f"Summarize the dialogue.\nDialogue: {ex['dialogue']}\nSummary:", max_new_tokens=60))
        refs.append(ex["summary"])
    return round(rouge_metric.compute(predictions=preds, references=refs)["rougeL"], 3)

def eval_nli(model):
    correct = 0
    for ex in nli_eval_raw:
        pred  = generate(model, f"Premise: {ex['premise']}\nHypothesis: {ex['hypothesis']}\nRelation:", max_new_tokens=5).lower()
        correct += int(NLI_LABELS[ex["label"]] in pred)
    return round(correct / len(nli_eval_raw), 3)

print("Evaluation functions defined.")

In [ ]:
MODELS_2TASK = {
    "emotion_specialist": "state_dicts/sd_emotion.pt",
    "summary_specialist": "state_dicts/sd_summary.pt",
    "weight_average":     "state_dicts/sd_weight_avg_2task.pt",
    "task_arithmetic":    "state_dicts/sd_task_arith_2task.pt",
    "breadcrumbs":        "state_dicts/sd_breadcrumbs_2task.pt",
    "ties":               "state_dicts/sd_ties_2task.pt",
}

results_2task = {}
for name, path in MODELS_2TASK.items():
    print(f"\nEvaluating {name}...")
    model = load_model_from_sd(path)
    results_2task[name] = {
        "emotion_acc": eval_emotion(model),
        "rougeL":      eval_summary(model),
    }
    print(f"  emotion_acc={results_2task[name]['emotion_acc']}  rougeL={results_2task[name]['rougeL']}")
    del model; gc.collect(); torch.cuda.empty_cache()

df_2task = pd.DataFrame(results_2task).T
print("\n=== 2-Task Demo Results ===")
print(df_2task.to_string())

### 2-Task Results — Analysis

The specialist models are task-specific upper bounds: the emotion specialist maximizes emotion accuracy at the cost of ROUGE-L, the summary specialist maximizes ROUGE-L. The **multitask baseline** — jointly trained on both tasks — is the practical reference that shows what supervised mixing achieves; merged models aim to match or exceed it without any mixed-task training data.

**Comparing merging methods against each other:**

**Weight Averaging and Task Arithmetic (λ=0.5) are mathematically identical** with two equally-weighted tasks — averaging fine-tuned weights is algebraically the same as applying task vectors at λ=0.5. Both achieve 0.782 emotion / 0.321 ROUGE-L, beating the multitask baseline on ROUGE-L (+0.020) while remaining below it on emotion (−0.100). This is the strongest baseline for any merged model.

**Breadcrumbs (d=0.2)** drops well below Weight Average on emotion (0.680 vs 0.782, −13%), with ROUGE-L matching only the multitask baseline (0.301 vs 0.301). Very aggressive pruning (keeping only 20% of parameters) removes too much task-relevant signal at λ=0.5 — the method underperforms its own baseline. Higher density is likely needed.

**TIES** is the worst merger on both metrics (0.636 emotion / 0.286 ROUGE-L), sitting −18.7% below Weight Average on emotion and −11% below on ROUGE-L. With only two tasks, majority sign voting is undefined: every parameter has exactly one vote per sign, yielding ~50% conflicts resolved arbitrarily. This is a structural limitation of TIES when N=2, independent of λ or density. I formulated the hypothesis that **adding a third task (enabling a proper 2-vs-1 majority vote) should unlock TIES's advantage**, and confirmed it was worth investigating in my next lab session. Before testing that, I first wanted to see whether parameter tuning could improve Task Arithmetic and Breadcrumbs.

**Full experiment results — Llama-3.2-3B-Instruct (our server):**

| Model | emotion_acc | ROUGE-L |
|---|---|---|
| base (zero-shot) | 0.318 | 0.181 |
| emotion specialist | 0.912 | 0.193 |
| summary specialist | 0.408 | 0.298 |
| multitask baseline | 0.882 | 0.301 |
| weight average | 0.782 | 0.321 |
| task arithmetic (λ=0.5) | 0.782 | 0.321 |
| breadcrumbs (λ=0.5, d=0.2) | 0.680 | 0.301 |
| TIES (λ=0.5, d=0.2) | 0.636 | 0.286 |

At full scale the ranking is identical: WA = TA > Breadcrumbs > TIES. All merged models outperform the zero-shot base on emotion, and the top two (WA/TA) improve ROUGE-L by +0.020 above the multitask baseline — demonstrating a concrete advantage of merging over joint training on this summarization metric.

---
## Part 2: Parameter Sweep — Task Arithmetic (λ) and Breadcrumbs (density)

Analyzing the initial results, I identified two directions to investigate and brought my interpretations to my TA during the lab session, who confirmed they were the right questions to pursue:

1. **Why does TIES fail with 2 tasks?** — the sign-conflict analysis above gives the theoretical answer; adding a third task is the planned fix (Part 3).
2. **Can we improve Task Arithmetic and Breadcrumbs through parameter tuning?** — the default λ=0.5 and density=0.2 are conservative starting points. A sweep over both could reveal better configurations and clarify the emotion/ROUGE-L trade-off.

**What λ controls (Task Arithmetic & Breadcrumbs):** λ scales the sum of task vectors before adding it to the base model. Low λ → *under-merging*: the model stays close to the base and task-specific performance is weak. High λ → stronger task signal but growing cross-task interference, eventually causing *catastrophic forgetting* — both tasks degrade as the summed task vectors overwhelm the base model's pre-trained representations. There is typically a performance peak followed by collapse as λ increases beyond a threshold.

**What density controls (Breadcrumbs):** density is the fraction of parameters *kept* per task vector, selected by absolute magnitude. Low density → aggressive pruning, retaining only the strongest task-specific entries. High density → approaches Task Arithmetic (density=1.0 is identical, no pruning at all). The rationale: high-magnitude parameters carry the task-specific signal; zeroing the rest reduces the cross-task noise introduced when multiple task vectors are summed. The risk of over-pruning (very low density): useful signal is discarded along with the noise.

In the demo we sweep λ ∈ {0.3, 0.7, 1.0} for Task Arithmetic and density ∈ {0.2, 0.5, 0.8} for Breadcrumbs (λ fixed at 0.5). The full experiment used λ ∈ {0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0} and density ∈ {0.2, 0.3, 0.4, 0.5, 0.6, 0.8}.

In [ ]:
SWEEP_TA_LAMBDAS   = [0.3, 0.7, 1.0]
SWEEP_BC_DENSITIES = [0.2, 0.5, 0.8]

def run_v2_sweep():
    base_sd    = load_sd("state_dicts/sd_base.pt")
    emotion_sd = load_sd("state_dicts/sd_emotion.pt")
    summary_sd = load_sd("state_dicts/sd_summary.pt")
    specialists = [emotion_sd, summary_sd]

    for lam in SWEEP_TA_LAMBDAS:
        lam_s = str(lam).replace(".", "")
        fname = f"sd_task_arith_l{lam_s}.pt"
        path  = f"state_dicts/{fname}"
        if not os.path.exists(path):
            print(f"Task Arithmetic λ={lam} → {fname}")
            sd = task_arithmetic(base_sd, specialists, lam=lam)
            torch.save(sd, path)
            del sd; gc.collect()
        else:
            print(f"  {fname} already exists — skipping.")

    for d in SWEEP_BC_DENSITIES:
        d_s   = str(d).replace(".", "")
        fname = f"sd_breadcrumbs_d{d_s}.pt"
        path  = f"state_dicts/{fname}"
        if not os.path.exists(path):
            print(f"Breadcrumbs density={d} → {fname}")
            sd = breadcrumbs(base_sd, specialists, lam=0.5, density=d)
            torch.save(sd, path)
            del sd; gc.collect()
        else:
            print(f"  {fname} already exists — skipping.")

    del base_sd, emotion_sd, summary_sd; gc.collect()
    print("Sweep complete.")

run_v2_sweep()

In [ ]:
SWEEP_MODELS = {}
for lam in SWEEP_TA_LAMBDAS:
    lam_s = str(lam).replace(".", "")
    SWEEP_MODELS[f"task_arith (λ={lam})"] = f"state_dicts/sd_task_arith_l{lam_s}.pt"
for d in SWEEP_BC_DENSITIES:
    d_s = str(d).replace(".", "")
    SWEEP_MODELS[f"breadcrumbs (d={d})"] = f"state_dicts/sd_breadcrumbs_d{d_s}.pt"

results_sweep = {}
for name, path in SWEEP_MODELS.items():
    print(f"\nEvaluating {name}...")
    model = load_model_from_sd(path)
    results_sweep[name] = {
        "emotion_acc": eval_emotion(model),
        "rougeL":      eval_summary(model),
    }
    print(f"  emotion_acc={results_sweep[name]['emotion_acc']}  rougeL={results_sweep[name]['rougeL']}")
    del model; gc.collect(); torch.cuda.empty_cache()

df_sweep = pd.DataFrame(results_sweep).T
print("\n=== Parameter Sweep Demo Results ===")
print(df_sweep.to_string())

### Parameter Sweep Results — Analysis

**Task Arithmetic — emotion/ROUGE-L trade-off:** Emotion accuracy peaks at λ=0.7 (0.842) — a +7.7% gain over the default λ=0.5 (0.782) — while ROUGE-L drops from 0.321 to 0.314 (−2.2%). As λ increases, the combined task vector pulls the model more aggressively toward both tasks simultaneously; since the emotion task vector is larger, emotion gains more. ROUGE-L suffers because the summarization direction is partially overridden. The best balanced point is **λ=0.6**: emotion=0.830 (+6.1%), ROUGE-L=0.315 (−1.9%) — both improved relative to the multitask baseline on ROUGE-L while approaching it on emotion. At λ=0.8 emotion drops back to 0.798, signaling the onset of interference — the emotion specialist's knowledge is no longer cleanly separable at this scale. Continued increase toward the *collapse threshold* (~λ=1.5 in preliminary estimates) would cause both tasks to degrade toward random as the task vectors overwhelm the base model's pre-trained representations (catastrophic forgetting of the base).

**Breadcrumbs — density improves ROUGE-L, not emotion:** At d=0.2, Breadcrumbs underperforms Task Arithmetic on both metrics (0.680 vs 0.782 emotion; 0.301 vs 0.321 ROUGE-L) — over-pruning discards task signal. As density rises, ROUGE-L improves monotonically up to d=0.5 (0.323), then plateaus. Emotion also rises, but at d=0.8 Breadcrumbs converges to Task Arithmetic (0.782 emotion / 0.321 ROUGE-L), confirming density=0.8 ≈ no pruning. Comparing **Breadcrumbs (d=0.5) vs Task Arithmetic (λ=0.5)**: Breadcrumbs gives −0.016 emotion (0.766 vs 0.782) in exchange for +0.002 ROUGE-L (0.323 vs 0.321). Small trade-off, but Breadcrumbs is the only method that surpasses Task Arithmetic on ROUGE-L. Comparing **Breadcrumbs (d=0.5) vs Task Arithmetic (λ=0.7)** — the best emotion config — the gap widens: −0.076 emotion (0.766 vs 0.842) for +0.009 ROUGE-L (0.323 vs 0.314). Breadcrumbs clearly favors summarization quality over emotion accuracy.

**Full experiment results — Task Arithmetic λ sweep (Llama-3.2-3B-Instruct):**

| Config | emotion_acc | ROUGE-L | vs WA/TA (λ=0.5) |
|---|---|---|---|
| task arithmetic λ=0.3 | 0.610 | 0.288 | −21.9% emotion, −10.3% ROUGE-L |
| task arithmetic λ=0.4 | 0.696 | 0.312 | −11.0% emotion, −2.8% ROUGE-L |
| task arithmetic λ=0.5 *(baseline)* | 0.782 | 0.321 | — |
| task arithmetic λ=0.6 | **0.830** | 0.315 | +6.1% emotion, −1.9% ROUGE-L |
| task arithmetic λ=0.7 | **0.842** | 0.314 | +7.7% emotion, −2.2% ROUGE-L |
| task arithmetic λ=0.8 | 0.798 | 0.309 | +2.0% emotion, −3.7% ROUGE-L |
| task arithmetic λ=1.0 | 0.838 | 0.311 | +7.2% emotion, −3.1% ROUGE-L |

**Full experiment results — Breadcrumbs density sweep (λ=0.5 fixed):**

| Config | emotion_acc | ROUGE-L | vs Task Arith (λ=0.5) |
|---|---|---|---|
| breadcrumbs d=0.2 | 0.680 | 0.301 | −13.0% emotion, −6.2% ROUGE-L |
| breadcrumbs d=0.3 | 0.714 | 0.311 | −8.7% emotion, −3.1% ROUGE-L |
| breadcrumbs d=0.4 | 0.754 | 0.319 | −3.6% emotion, −0.6% ROUGE-L |
| breadcrumbs d=0.5 | 0.766 | **0.323** | −2.0% emotion, **+0.6% ROUGE-L** |
| breadcrumbs d=0.6 | 0.780 | 0.319 | −0.3% emotion, −0.6% ROUGE-L |
| breadcrumbs d=0.8 | 0.782 | 0.321 | ≈Task Arithmetic (density → 1.0) |

**Key takeaway:** Task Arithmetic and Breadcrumbs expose a clear emotion/ROUGE-L trade-off that can be controlled through their parameters. No single configuration dominates on both metrics simultaneously — the best choice depends on which task is prioritized. We now turn to testing whether adding a third task resolves the TIES limitation.

---
## Part 3: Three-Task Experiment — Adding NLI

The sign-conflict analysis from Part 1 predicts that TIES should improve with three tasks: a 2-vs-1 majority vote is well-defined and can deterministically resolve conflicts. We add a third specialist trained on **Natural Language Inference** (`nyu-mll/multi_nli`) — predicting whether a premise *entails*, is *neutral to*, or *contradicts* a hypothesis.

With three task vectors, all four merging methods are re-run and a third metric (NLI accuracy) is added to evaluation.

In [ ]:
print("Fine-tuning NLI specialist...")
finetune(nli_train_ds, "adapters/nli")

In [ ]:
linearize("adapters/nli", "state_dicts/sd_nli.pt")

def run_3task_merges():
    base_sd    = load_sd("state_dicts/sd_base.pt")
    emotion_sd = load_sd("state_dicts/sd_emotion.pt")
    summary_sd = load_sd("state_dicts/sd_summary.pt")
    nli_sd     = load_sd("state_dicts/sd_nli.pt")
    specialists = [emotion_sd, summary_sd, nli_sd]

    configs = [
        ("sd_weight_avg_3task.pt",  lambda: weight_average(specialists)),
        ("sd_task_arith_3task.pt",  lambda: task_arithmetic(base_sd, specialists, lam=0.5)),
        ("sd_breadcrumbs_3task.pt", lambda: breadcrumbs(base_sd, specialists, lam=0.5, density=0.5)),
        ("sd_ties_3task.pt",        lambda: ties(base_sd, specialists, lam=0.5, density=0.4)),
    ]
    for fname, merge_fn in configs:
        path = f"state_dicts/{fname}"
        if not os.path.exists(path):
            print(f"Merging → {fname}")
            sd = merge_fn()
            torch.save(sd, path)
            del sd; gc.collect()
        else:
            print(f"  {fname} already exists — skipping.")

    del base_sd, emotion_sd, summary_sd, nli_sd; gc.collect()
    print("3-task merges complete.")

run_3task_merges()

In [ ]:
MODELS_3TASK = {
    "emotion_specialist": "state_dicts/sd_emotion.pt",
    "summary_specialist": "state_dicts/sd_summary.pt",
    "nli_specialist":     "state_dicts/sd_nli.pt",
    "weight_average":     "state_dicts/sd_weight_avg_3task.pt",
    "task_arithmetic":    "state_dicts/sd_task_arith_3task.pt",
    "breadcrumbs":        "state_dicts/sd_breadcrumbs_3task.pt",
    "ties":               "state_dicts/sd_ties_3task.pt",
}

results_3task = {}
for name, path in MODELS_3TASK.items():
    print(f"\nEvaluating {name}...")
    model = load_model_from_sd(path)
    results_3task[name] = {
        "emotion_acc": eval_emotion(model),
        "nli_acc":     eval_nli(model),
        "rougeL":      eval_summary(model),
    }
    print(f"  emotion={results_3task[name]['emotion_acc']}  nli={results_3task[name]['nli_acc']}  rougeL={results_3task[name]['rougeL']}")
    del model; gc.collect(); torch.cuda.empty_cache()

df_3task = pd.DataFrame(results_3task).T
print("\n=== 3-Task Demo Results ===")
print(df_3task.to_string())

### 3-Task Results — Analysis

Adding NLI changes the merging dynamics considerably. Each merger now combines three task vectors, and NLI accuracy becomes a third dimension of evaluation.

**Comparing merging methods against each other:**

**Task Arithmetic** is the best 3-task merger overall. It achieves the highest NLI accuracy by far (0.834 vs 0.680 for Weight Average, +22.6%), confirming that task vectors capture the strong NLI task direction more faithfully than raw weight averaging. Compared to Weight Average, it also marginally improves emotion (0.630 vs 0.622, +1.3%) with identical ROUGE-L (0.298 vs 0.297). The cost: adding NLI as a third task vector has pulled emotion down significantly compared to the 2-task experiment (0.630 vs 0.782, −19%), as the three task vectors now sum to a larger total perturbation that competes against the emotion direction.

**Breadcrumbs** is the closest to Task Arithmetic. It edges Task Arithmetic on emotion (0.636 vs 0.630, +1%) but falls behind on NLI (0.814 vs 0.834, −2.4%) and ROUGE-L (0.291 vs 0.298, −2.3%). The d=0.5 pruning helps emotion slightly — sparse merging removes some of the noise from the NLI and summary vectors that would otherwise interfere with the emotion direction — but at the cost of NLI accuracy: the NLI task vector has useful spread across many parameters, and pruning discards some of it.

**Weight Averaging** loses the most compared to Task Arithmetic, especially on NLI (−22.6%). Direct weight averaging dilutes the strong NLI task direction because the absolute magnitude of fine-tuned weights is smaller than the corresponding task vector — the NLI specialist's weights are averaged down with the base model's weights via the other two specialists.

**TIES (λ=0.5)** is still the worst merger despite three tasks enabling a proper 2-vs-1 majority vote. It is −7.9% below Task Arithmetic on emotion (0.580 vs 0.630), −19.4% below on NLI (0.672 vs 0.834), and only matches on ROUGE-L (0.298). Sign conflicts are now theoretically resolved, so the failure must have a different cause. Analyzing the TIES mechanism, I identified the likely reason: because TIES zeros the *disagreeing* subset of parameters before averaging, the effective magnitude of the merged task vector is intrinsically smaller than in Task Arithmetic at the same λ. The model is under-merged — **a higher λ is needed to compensate.** I brought this hypothesis to my TA, who confirmed it was consistent with the theory and worth testing with a systematic sweep.

**Full experiment results — 3-task (Llama-3.2-3B-Instruct):**

| Model | emotion_acc | NLI acc | ROUGE-L |
|---|---|---|---|
| emotion specialist | 0.912 | 0.410 | 0.193 |
| summary specialist | 0.408 | 0.510 | 0.298 |
| NLI specialist | 0.238 | 0.858 | 0.205 |
| multitask baseline | 0.882 | 0.380 | 0.301 |
| weight average | 0.622 | 0.680 | 0.297 |
| task arithmetic (λ=0.5) | 0.630 | **0.834** | 0.298 |
| breadcrumbs (λ=0.5, d=0.5) | **0.636** | 0.814 | 0.291 |
| TIES (λ=0.5, d=0.4) | 0.580 | 0.672 | 0.298 |

All merged models comfortably exceed the multitask baseline on NLI (0.672–0.834 vs 0.380) — a major gain from incorporating the NLI specialist. The trade-off is a notable emotion drop compared to the 2-task experiment: the best 3-task merger (Task Arithmetic) achieves only 0.630 emotion vs 0.782 in the 2-task case, because three competing task vectors create more cross-task interference than two.

---
## Part 4: TIES Parameter Sweep

Analyzing the TIES mechanism, I identified that λ=0.5 was likely under-scaling the TIES task vectors: because TIES zeroes disagreeing parameters before averaging, the effective perturbation applied to the base model is intrinsically smaller than in Task Arithmetic at the same λ. I planned a systematic sweep over λ and density to test this, and my TA confirmed it was the right investigation. We sweep λ ∈ {0.5, 1.0} × density ∈ {0.2, 0.4} in the demo (the full experiment used a 4×4 grid: λ ∈ {0.3, 0.5, 0.7, 1.0} × density ∈ {0.2, 0.4, 0.6, 0.8}).

In [ ]:
SWEEP_LAMBDAS   = [0.5, 1.0]
SWEEP_DENSITIES = [0.2, 0.4]

def run_ties_sweep():
    base_sd    = load_sd("state_dicts/sd_base.pt")
    emotion_sd = load_sd("state_dicts/sd_emotion.pt")
    summary_sd = load_sd("state_dicts/sd_summary.pt")
    nli_sd     = load_sd("state_dicts/sd_nli.pt")
    specialists = [emotion_sd, summary_sd, nli_sd]

    for lam in SWEEP_LAMBDAS:
        for d in SWEEP_DENSITIES:
            lam_s = str(lam).replace(".", "")
            d_s   = str(d).replace(".", "")
            fname = f"sd_ties_v4_l{lam_s}_d{d_s}.pt"
            path  = f"state_dicts/{fname}"
            if not os.path.exists(path):
                print(f"TIES λ={lam}, density={d} → {fname}")
                sd = ties(base_sd, specialists, lam=lam, density=d)
                torch.save(sd, path)
                del sd; gc.collect()
            else:
                print(f"  {fname} already exists — skipping.")

    del base_sd, emotion_sd, summary_sd, nli_sd; gc.collect()
    print("TIES sweep complete.")

run_ties_sweep()

In [ ]:
results_v4 = {}
for lam in SWEEP_LAMBDAS:
    for d in SWEEP_DENSITIES:
        lam_s = str(lam).replace(".", "")
        d_s   = str(d).replace(".", "")
        path  = f"state_dicts/sd_ties_v4_l{lam_s}_d{d_s}.pt"
        name  = f"TIES λ={lam}, d={d}"
        print(f"\nEvaluating {name}...")
        model = load_model_from_sd(path)
        results_v4[name] = {
            "lambda":      lam,
            "density":     d,
            "emotion_acc": eval_emotion(model),
            "nli_acc":     eval_nli(model),
            "rougeL":      eval_summary(model),
        }
        print(f"  emotion={results_v4[name]['emotion_acc']}  nli={results_v4[name]['nli_acc']}  rougeL={results_v4[name]['rougeL']}")
        del model; gc.collect(); torch.cuda.empty_cache()

df_v4 = pd.DataFrame(results_v4).T
print("\n=== TIES Sweep Demo Results ===")
print(df_v4.to_string())

### TIES Sweep — Findings and Conclusion

**The sweep confirms the hypothesis: λ=0.5 was severely under-scaling TIES task vectors.** Increasing λ from 0.5 to 1.0 dramatically improves all three metrics — emotion +0.130, NLI +0.144, ROUGE-L +0.005 at d=0.2. This is the dominant effect; density is secondary.

**Comparing TIES (λ=1.0, d=0.2) against Task Arithmetic (λ=0.5):** TIES now leads on emotion (0.710 vs 0.630, +12.7%) and ROUGE-L (0.303 vs 0.298, +1.7%), while remaining slightly below on NLI (0.816 vs 0.834, −2.2%). Overall, TIES at its optimal configuration is the best single merger across the three metrics combined.

**Density trade-off at λ=1.0:** d=0.2 vs d=0.4 shows a NLI/emotion-ROUGE split — d=0.4 peaks on NLI (0.840, +0.024 vs d=0.2) but drops on emotion (0.682 vs 0.710, −3.9%) and ROUGE-L (0.283 vs 0.303, −7.1%). Sparser merging (d=0.2) retains more balanced performance across all three tasks; denser merging (d=0.4) specializes more toward NLI at the cost of the other two.

**Why does TIES tolerate high λ better than Task Arithmetic?** Task Arithmetic at high λ sums all task vectors regardless of agreement, which risks catastrophic forgetting — the combined perturbation can overwhelm the base model's pre-trained representations, causing degradation on both tasks beyond the collapse threshold. TIES's sign election acts as a natural regularizer: parameters where tasks conflict are zeroed (not applied at scale), while only parameters with broad agreement receive the full λ treatment. This limits the total perturbation even as λ increases, making TIES more robust at λ=1.0 than Task Arithmetic would be at the same scale. The cost of this mechanism is precisely what we observed: TIES under-merges at low λ and needs a higher value to achieve the same effective signal strength.

**Lower bound — λ=0.3:** At λ=0.3, TIES degrades to 0.478 emotion / 0.648 NLI / 0.269 ROUGE-L, well below even the zero-shot base on relative terms. The task vectors are scaled too weakly; after sign election zeroes the minority parameters, the remaining signal is too small to steer the model meaningfully. This confirms both a lower bound on λ for TIES and the under-merging behavior at small scaling factors.

**Full experiment results — TIES 4×4 sweep (Llama-3.2-3B-Instruct):**

| Config | emotion_acc | NLI acc | ROUGE-L |
|---|---|---|---|
| TIES λ=0.3, d=0.2 | 0.478 | 0.648 | 0.269 |
| TIES λ=0.3, d=0.4 | 0.492 | 0.650 | 0.275 |
| TIES λ=0.5, d=0.2 | 0.566 | 0.670 | 0.289 |
| TIES λ=0.5, d=0.4 *(v3 baseline)* | 0.580 | 0.672 | 0.298 |
| TIES λ=0.7, d=0.2 | 0.670 | 0.694 | 0.297 |
| TIES λ=0.7, d=0.4 | 0.680 | 0.714 | 0.297 |
| **TIES λ=1.0, d=0.2** | **0.710** | 0.816 | **0.303** |
| TIES λ=1.0, d=0.4 | 0.682 | **0.840** | 0.283 |
| *Task Arithmetic λ=0.5 (reference)* | *0.630* | *0.834* | *0.298* |

**Overall conclusion:** TIES requires two conditions to work well — (1) ≥3 tasks for meaningful sign election and (2) a calibrated λ to compensate for the reduced effective perturbation from sign-based filtering. Neither condition alone is sufficient. With both satisfied (λ=1.0, d=0.2), TIES becomes the best overall 3-task merger, outperforming Task Arithmetic on emotion and ROUGE-L while staying competitive on NLI.

All fine-tuned adapters, merged models, and the full codebase are available at:
- **GitHub:** https://github.com/islem-kms/COMP6861_MergeLLMs
- **HuggingFace:** https://huggingface.co/islemkms